In [1]:
import numpy as np
import time
import sys

from io import UnsupportedOperation
import pickle as pkl


# Add shared location for auxillary functions
sys.path.insert(1, './../../AuxillaryFunctions')

# import personal Functions
from GenerateClassDataFuncs import GenerateEvalData_ClemCafe
from GenerateClassDataFuncs import GenerateEvalData_OREBA
from EvaluationMethodFuncs import KyritEval_PtVsWindow
from EvaluationMethodFuncs import DongEval_PtVsPt
from EvaluationMethodFuncs import TimePoint2Window
from EvaluationMethodFuncs import Window2TimePoint
from EvaluationMethodFuncs import PrintStats_SingleLine


import csv



print("Finished importing libraries.")


Finished importing libraries.


In [2]:
### Demo of reading a file ###

# My_File = "./FiveFold_CTCResults_v1/Clem_Intake/fold0/Clemson_p410_c2_grm_std_uni.csv"
# My_File = "./Predictions/OHOv3_CTC_DefaultSplit_v0_B8000/OneHandOrebaV3_1005_1_grm_std_uni.csv"
My_File = "./Predictions/OHOv3_CTC_DefaultSplit_v1_B13500/OneHandOrebaV3_1005_1_grm_std_uni.csv"



def ReadPredictionsFileForDetections(My_File):
	with open(My_File, newline='') as csvfile:
		spamreader = csv.reader(csvfile, delimiter=',', quotechar='|')
		cnt=0
		Detections=[]
		for row in spamreader:
			cnt+=1
			if (row[4]!='1'): # Fifth Column is either 1 for null action or 2 for intake event
				# print(f"{cnt/8}")
				Detections.append(cnt/8)
			#end if
		#end for row
	#end with open(.csv)
	
	return Detections
# end of ReadPredictionsFileForDetections()# end of ReadPredictionsFileForDetections()

In [3]:
### Grab list of all files and their corresponding ground truth ###

ResultsFilesAllFolds = [
	[
		"Fold0_B19000/OneHandOrebaV3_1005_1_grm_std_uni.csv",  "Fold0_B19000/OneHandOrebaV3_1061_1_grm_std_uni.csv",
		"Fold0_B19000/OneHandOrebaV3_1011_1_grm_std_uni.csv",  "Fold0_B19000/OneHandOrebaV3_1072_1_grm_std_uni.csv",
		"Fold0_B19000/OneHandOrebaV3_1016_1_grm_std_uni.csv",  "Fold0_B19000/OneHandOrebaV3_1079_1_grm_std_uni.csv",
		"Fold0_B19000/OneHandOrebaV3_1021_1_grm_std_uni.csv",  "Fold0_B19000/OneHandOrebaV3_1084_1_grm_std_uni.csv",
		"Fold0_B19000/OneHandOrebaV3_1026_1_grm_std_uni.csv",  "Fold0_B19000/OneHandOrebaV3_1089_1_grm_std_uni.csv",
		"Fold0_B19000/OneHandOrebaV3_1031_1_grm_std_uni.csv",  "Fold0_B19000/OneHandOrebaV3_1094_1_grm_std_uni.csv",
		"Fold0_B19000/OneHandOrebaV3_1037_1_grm_std_uni.csv",  "Fold0_B19000/OneHandOrebaV3_1099_1_grm_std_uni.csv",
		"Fold0_B19000/OneHandOrebaV3_1044_1_grm_std_uni.csv",  "Fold0_B19000/OneHandOrebaV3_1104_1_grm_std_uni.csv",
		"Fold0_B19000/OneHandOrebaV3_1050_1_grm_std_uni.csv",  "Fold0_B19000/OneHandOrebaV3_1110_1_grm_std_uni.csv",
		"Fold0_B19000/OneHandOrebaV3_1055_1_grm_std_uni.csv",  "Fold0_B19000/OneHandOrebaV3_1116_1_grm_std_uni.csv"
	],
	[
		"Fold1_B13500/OneHandOrebaV3_1001_1_grm_std_uni.csv",  "Fold1_B13500/OneHandOrebaV3_1056_1_grm_std_uni.csv",
		"Fold1_B13500/OneHandOrebaV3_1006_1_grm_std_uni.csv",  "Fold1_B13500/OneHandOrebaV3_1063_1_grm_std_uni.csv",
		"Fold1_B13500/OneHandOrebaV3_1012_1_grm_std_uni.csv",  "Fold1_B13500/OneHandOrebaV3_1073_1_grm_std_uni.csv",
		"Fold1_B13500/OneHandOrebaV3_1017_1_grm_std_uni.csv",  "Fold1_B13500/OneHandOrebaV3_1080_1_grm_std_uni.csv",
		"Fold1_B13500/OneHandOrebaV3_1022_1_grm_std_uni.csv",  "Fold1_B13500/OneHandOrebaV3_1085_1_grm_std_uni.csv",
		"Fold1_B13500/OneHandOrebaV3_1027_1_grm_std_uni.csv",  "Fold1_B13500/OneHandOrebaV3_1090_1_grm_std_uni.csv",
		"Fold1_B13500/OneHandOrebaV3_1032_1_grm_std_uni.csv",  "Fold1_B13500/OneHandOrebaV3_1095_1_grm_std_uni.csv",
		"Fold1_B13500/OneHandOrebaV3_1039_1_grm_std_uni.csv",  "Fold1_B13500/OneHandOrebaV3_1100_1_grm_std_uni.csv",
		"Fold1_B13500/OneHandOrebaV3_1045_1_grm_std_uni.csv",  "Fold1_B13500/OneHandOrebaV3_1105_1_grm_std_uni.csv",
		"Fold1_B13500/OneHandOrebaV3_1051_1_grm_std_uni.csv",  "Fold1_B13500/OneHandOrebaV3_1111_1_grm_std_uni.csv"
	],
	[
		"Fold2_B26500/OneHandOrebaV3_1002_1_grm_std_uni.csv",  "Fold2_B26500/OneHandOrebaV3_1057_1_grm_std_uni.csv",
		"Fold2_B26500/OneHandOrebaV3_1007_1_grm_std_uni.csv",  "Fold2_B26500/OneHandOrebaV3_1064_1_grm_std_uni.csv",
		"Fold2_B26500/OneHandOrebaV3_1013_1_grm_std_uni.csv",  "Fold2_B26500/OneHandOrebaV3_1075_1_grm_std_uni.csv",
		"Fold2_B26500/OneHandOrebaV3_1018_1_grm_std_uni.csv",  "Fold2_B26500/OneHandOrebaV3_1081_1_grm_std_uni.csv",
		"Fold2_B26500/OneHandOrebaV3_1023_1_grm_std_uni.csv",  "Fold2_B26500/OneHandOrebaV3_1086_1_grm_std_uni.csv",
		"Fold2_B26500/OneHandOrebaV3_1028_1_grm_std_uni.csv",  "Fold2_B26500/OneHandOrebaV3_1091_1_grm_std_uni.csv",
		"Fold2_B26500/OneHandOrebaV3_1033_1_grm_std_uni.csv",  "Fold2_B26500/OneHandOrebaV3_1096_1_grm_std_uni.csv",
		"Fold2_B26500/OneHandOrebaV3_1040_1_grm_std_uni.csv",  "Fold2_B26500/OneHandOrebaV3_1101_1_grm_std_uni.csv",
		"Fold2_B26500/OneHandOrebaV3_1046_1_grm_std_uni.csv",  "Fold2_B26500/OneHandOrebaV3_1107_1_grm_std_uni.csv",
		"Fold2_B26500/OneHandOrebaV3_1052_1_grm_std_uni.csv",  "Fold2_B26500/OneHandOrebaV3_1112_1_grm_std_uni.csv"
	],
	[
		"Fold3_B19000/OneHandOrebaV3_1003_1_grm_std_uni.csv",  "Fold3_B19000/OneHandOrebaV3_1059_1_grm_std_uni.csv",
		"Fold3_B19000/OneHandOrebaV3_1008_1_grm_std_uni.csv",  "Fold3_B19000/OneHandOrebaV3_1067_1_grm_std_uni.csv",
		"Fold3_B19000/OneHandOrebaV3_1014_1_grm_std_uni.csv",  "Fold3_B19000/OneHandOrebaV3_1076_1_grm_std_uni.csv",
		"Fold3_B19000/OneHandOrebaV3_1019_1_grm_std_uni.csv",  "Fold3_B19000/OneHandOrebaV3_1082_1_grm_std_uni.csv",
		"Fold3_B19000/OneHandOrebaV3_1024_1_grm_std_uni.csv",  "Fold3_B19000/OneHandOrebaV3_1087_1_grm_std_uni.csv",
		"Fold3_B19000/OneHandOrebaV3_1029_1_grm_std_uni.csv",  "Fold3_B19000/OneHandOrebaV3_1092_1_grm_std_uni.csv",
		"Fold3_B19000/OneHandOrebaV3_1035_1_grm_std_uni.csv",  "Fold3_B19000/OneHandOrebaV3_1097_1_grm_std_uni.csv",
		"Fold3_B19000/OneHandOrebaV3_1041_1_grm_std_uni.csv",  "Fold3_B19000/OneHandOrebaV3_1102_1_grm_std_uni.csv",
		"Fold3_B19000/OneHandOrebaV3_1047_1_grm_std_uni.csv",  "Fold3_B19000/OneHandOrebaV3_1108_1_grm_std_uni.csv",
		"Fold3_B19000/OneHandOrebaV3_1053_1_grm_std_uni.csv",  "Fold3_B19000/OneHandOrebaV3_1113_1_grm_std_uni.csv"
	],
	[
		"Fold4_B8500/OneHandOrebaV3_1004_1_grm_std_uni.csv",  "Fold4_B8500/OneHandOrebaV3_1060_1_grm_std_uni.csv",
		"Fold4_B8500/OneHandOrebaV3_1010_1_grm_std_uni.csv",  "Fold4_B8500/OneHandOrebaV3_1068_1_grm_std_uni.csv",
		"Fold4_B8500/OneHandOrebaV3_1015_1_grm_std_uni.csv",  "Fold4_B8500/OneHandOrebaV3_1077_1_grm_std_uni.csv",
		"Fold4_B8500/OneHandOrebaV3_1020_1_grm_std_uni.csv",  "Fold4_B8500/OneHandOrebaV3_1083_1_grm_std_uni.csv",
		"Fold4_B8500/OneHandOrebaV3_1025_1_grm_std_uni.csv",  "Fold4_B8500/OneHandOrebaV3_1088_1_grm_std_uni.csv",
		"Fold4_B8500/OneHandOrebaV3_1030_1_grm_std_uni.csv",  "Fold4_B8500/OneHandOrebaV3_1093_1_grm_std_uni.csv",
		"Fold4_B8500/OneHandOrebaV3_1036_1_grm_std_uni.csv",  "Fold4_B8500/OneHandOrebaV3_1098_1_grm_std_uni.csv",
		"Fold4_B8500/OneHandOrebaV3_1043_1_grm_std_uni.csv",  "Fold4_B8500/OneHandOrebaV3_1103_1_grm_std_uni.csv",
		"Fold4_B8500/OneHandOrebaV3_1048_1_grm_std_uni.csv",  "Fold4_B8500/OneHandOrebaV3_1109_1_grm_std_uni.csv",
		"Fold4_B8500/OneHandOrebaV3_1054_1_grm_std_uni.csv",  "Fold4_B8500/OneHandOrebaV3_1115_1_grm_std_uni.csv"
	]
]

print("Finished Defining Fold Filepaths.")

Finished Defining Fold Filepaths.


In [4]:
##### Grab GT Values from the Pickle Database

DATABASE_FILEPATH = "./../../Pickle_Databases/OHO_Dom_v2.pkl"

with open(DATABASE_FILEPATH,'rb') as fh:
	dataset = pkl.load(fh)
#OREBA_Cucumber={'UniqueID','proc_data','handedness','bites_gt'}


print("Finished importing OREBA (Dom Only GT) Pickle.")


Finished importing OREBA (Dom Only GT) Pickle.


In [12]:
#### DEFINE CONSTANTS WHICH CAN BE ADJUSTED


# Float value used to determine how many sec the detection can be away from teh window
WIN_TOLERANCE = 99
EvalSelect = 1 # 1 = Dong Eval, 2 = Kyritsis Window Tol

USE_FIVEFOLD_FLAG = True
USE_DEFAULTSPLIT_FLAG = False

# PredDirPath = './Predictions/OHOv3_CTC_DefaultSplit_v0_B8000/'
PredDirPath = './FiveFold_CTCResults_v1/OHOv3_Intake/'


In [13]:
######################
#### ONLY RUN WITH FIVE FOLD VALIDATION AND NOT DEFAULT SPLIT
######################

if USE_FIVEFOLD_FLAG == True:

	# Global Results from all folds (each element is a list from each fold)
	All_All_TP = []
	All_All_FP = []
	All_All_FN = []

	
	allIDs=dataset['UniqueID']
	

	for currFold in range(len(ResultsFilesAllFolds)):

		# Fold number to evaluate. Valid range is [0,4]
		FOLD_SELECT = currFold
		# Directory containing all results for the specified fold

		# Select Current Fold Results Filenames
		ResultsFiles = ResultsFilesAllFolds[FOLD_SELECT]
		# Create Sanity Check of All Participant and Meal IDs from filenames 
		#     to check against pickle Unique IDs later
		mealID_Check = []
		for filename in ResultsFiles:
			filenameComponents = filename.split("_") 
			mealID_Check.append(filenameComponents[-5]+ "_" + filenameComponents[-4])
		# end of for filename


		# Define blank sets for results
		All_TP=[]
		All_FP=[]
		All_FN=[]
		

		for idx in range(0, len(ResultsFiles)):
			try:
				pickle_idx = allIDs.index(mealID_Check[idx])
			except ValueError:
				print("ERROR: Mismatched FileID ({}) not found in list of Unique IDs.".format(currFileID))
				continue # skip to next file
			# end of try except
			
			currFilename = PredDirPath + ResultsFiles[idx]

			currDets = np.array(ReadPredictionsFileForDetections(currFilename))
			if EvalSelect==1: # Using Dong Eval
				currGT = Window2TimePoint(dataset['bites_gt'][pickle_idx])
			elif EvalSelect==2: # Using Kyritsis Eval with tolerance
				currGT = dataset['bites_gt'][pickle_idx]
			#end EvalSelect switch

			currPickleID = dataset['UniqueID'][pickle_idx]
			currFileID = mealID_Check[idx]
			if currPickleID != currFileID:
				print("ERROR: Mismatched FileID ({}) and MealID ({}).".format(currFileID, currPickleID))
			# end of Pickle ID Sanity check


			if EvalSelect==1: # Using Dong Eval
				[TP, FP, FN, _] = DongEval_PtVsPt(currDets, currGT)
			elif EvalSelect==2: # Using Kyritsis Eval with tolerance
				[TP, FP, FN, FP1, FP2, _] = KyritEval_PtVsWindow(currDets, currGT, WIN_TOLERANCE=WIN_TOLERANCE)
				# [TP, FP, FN, FP1, FP2, Key] = KyritEval_PtVsWindow(currDets, currGT, WIN_TOLERANCE = 0.0, BEFORE_TOL = 0.0, AFTER_TOL = 0.0):
			# end of EvalSelect switch 
			All_TP.append(TP)
			All_FP.append(FP)
			All_FN.append(FN)

			# PrintStats_SingleLine(TP, FP, FN)
		# end of for idx

		All_All_TP.append(All_TP)
		All_All_FP.append(All_FP)
		All_All_FN.append(All_FN)

	# end of for fold

	print("Complete Evaluating Meals.")
else:
	print("Data is not five fold, so no data was evaluated.")
# end of if USE_FIVEFOLD FLAG

Complete Evaluating Meals.


In [14]:
######################
#### ONLY RUN WITH DEFAULT SPLIT AND NOT FIVE FOLD VALIDATION
######################

if USE_DEFAULTSPLIT_FLAG == True:

	# Global Results from all folds (each element is a list from each fold)
	All_All_TP = []
	All_All_FP = []
	All_All_FN = []


	for currFold in range(len(ResultsFilesAllFolds)):

		# Fold number to evaluate. Valid range is [0,4]
		FOLD_SELECT = currFold
		# Select Current Fold Results Filenames
		ResultsFiles = ResultsFilesAllFolds[FOLD_SELECT]
		# Create Sanity Check of All Participant and Meal IDs from filenames 
		#     to check against pickle Unique IDs later
		mealID_Check = []
		for filename in ResultsFiles:
			filenameComponents = filename.split("_") 
			mealID_Check.append(filenameComponents[-5]+ "_" + filenameComponents[-4])
		# end of for filename


		# Define blank sets for results
		All_TP=[]
		All_FP=[]
		All_FN=[]

		allIDs=dataset['UniqueID']

		for idx in range(0, len(ResultsFiles)):
			try:
				pickle_idx = allIDs.index(mealID_Check[idx])
			except ValueError:
				print("ERROR: Mismatched FileID ({}) not found in list of Unique IDs.".format(currFileID))
				continue # skip to next file
			# end of try except

			currFilename = PredDirPath + ResultsFiles[idx]

			currDets = np.array(ReadPredictionsFileForDetections(currFilename))
			if EvalSelect==1: # Using Dong Eval
				currGT = Window2TimePoint(dataset['bites_gt'][pickle_idx])
			elif EvalSelect==2: # Using Kyritsis Eval with tolerance
				currGT = dataset['bites_gt'][pickle_idx]
			#end EvalSelect switch

			currPickleID = dataset['UniqueID'][pickle_idx]
			currFileID = mealID_Check[idx]
			if currPickleID != currFileID:
				print("ERROR: Mismatched FileID ({}) and MealID ({}).".format(currFileID, currPickleID))
				continue
			# end of Pickle ID Sanity check


			if EvalSelect==1: # Using Dong Eval
				[TP, FP, FN, _] = DongEval_PtVsPt(currDets, currGT)
			elif EvalSelect==2: # Using Kyritsis Eval with tolerance
				[TP, FP, FN, FP1, FP2, _] = KyritEval_PtVsWindow(currDets, currGT, WIN_TOLERANCE=WIN_TOLERANCE)
				# [TP, FP, FN, FP1, FP2, Key] = KyritEval_PtVsWindow(currDets, currGT, WIN_TOLERANCE = 0.0, BEFORE_TOL = 0.0, AFTER_TOL = 0.0):
			# end of EvalSelect switch 
			All_TP.append(TP)
			All_FP.append(FP)
			All_FN.append(FN)

			# PrintStats_SingleLine(TP, FP, FN)
		# end of for idx

		All_All_TP.append(All_TP)
		All_All_FP.append(All_FP)
		All_All_FN.append(All_FN)

	# end of for fold

	print("Complete Evaluating Meals.")
else:
	print("Data is not Default Split, so no data was evaluated.")
# end of if USE_DEFAULTSPLIT_FLAG


Data is not Default Split, so no data was evaluated.


In [15]:
#### Caclulate total performance on data

Final_TP = 0
Final_FP = 0
Final_FN = 0

for i in range(len(ResultsFilesAllFolds)):
	Final_TP += sum(All_All_TP[i]) 
	Final_FP += sum(All_All_FP[i]) 
	Final_FN += sum(All_All_FN[i])
# end of for fold loop 

if EvalSelect == 1:
	print("Dong Eval Pt2Pt")
elif EvalSelect == 2:
	print("Kyritsis Eval Pt2Window {} sec Tol".format(WIN_TOLERANCE))
else:
	print("Unknown Eval Selection.")
# end of eval print

PrintStats_SingleLine(Final_TP, Final_FP, Final_FN,VerbosePrintOpt=1)


Dong Eval Pt2Pt
F1     TPR    PPV    TP     FP     FN    
90.248 87.271 93.435 3003   211    438   
